In [1]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


## Clean Dataset #1

Visual Layer settings:
- Mislabels: 0-1
- Duplicates: 0.9159

In [1]:
from pathlib import Path
import json
import csv
import os
import shutil
from typing import Dict, List, Set, Tuple
from tqdm.notebook import tqdm
from tqdm import tqdm


#local paths

VL_JSON = Path("/Users/saeedarellano/Desktop/imagenet_jsons/metadata_11.json")
IMAGENET_ROOT = Path("/Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC")
OUT_ROOT = Path("/Users/saeedarellano/visual_layer/capstone_project_Visual-Layer/clean")

# Policy settings
UNIQUENESS_THRESHOLD = 999  # Keep images with score >= this value
KEEP_TOPK_PER_CLUSTER = None  # Set to int (e.g., 5) to keep top-K per cluster, or None for threshold mode
DROP_CLUSTERS_FILE = None  # Path to file with cluster IDs to drop, or None
KEEP_UNSEEN = True  # True = keep images not in VL export, False = drop them
MODE = "symlink"  # "symlink" or "copy"
DRY_RUN = True  # Set to False to actually create files
INCLUDE_VAL = True  # Set to True to also process val/ split

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_vl_exports(paths: List[Path]) -> List[dict]:
    """Load and merge Visual Layer export JSON files."""
    items = []
    for p in paths:
        print(f"Loading {p.name}...")
        with p.open("r") as f:
            data = json.load(f)
        if "media_items" not in data:
            raise ValueError(f"{p} missing 'media_items' field")
        items.extend(data["media_items"])
    print(f"✓ Loaded {len(items):,} total items from {len(paths)} file(s)")
    return items


def build_keep_set(
    vl_items: List[dict],
    uniqueness_threshold: float,
    keep_topk_per_cluster: int | None,
    dropped_clusters: Set[str],
) -> Tuple[Set[str], Set[str], Dict]:
    """Return (keep_filenames, drop_filenames, stats) based on policy."""
    stats = {"clusters_dropped": len(dropped_clusters)}
    
    if keep_topk_per_cluster is not None:
        # Top-K per cluster mode
        by_cluster: Dict[str, List[dict]] = {}
        for it in vl_items:
            cid = it.get("cluster_id", "NO_CLUSTER")
            by_cluster.setdefault(cid, []).append(it)

        keep, drop = set(), set()
        cluster_stats = {}
        
        for cid, group in by_cluster.items():
            if cid in dropped_clusters:
                for it in group:
                    fn = it.get("file_name")
                    if fn:
                        drop.add(fn)
                cluster_stats[cid] = {"total": len(group), "kept": 0, "dropped": len(group)}
                continue

            group_sorted = sorted(group, key=lambda x: x.get("uniqueness_score", 0.0), reverse=True)
            kept_count = min(keep_topk_per_cluster, len(group_sorted))
            
            for it in group_sorted[:kept_count]:
                fn = it.get("file_name")
                if fn:
                    keep.add(fn)
            for it in group_sorted[kept_count:]:
                fn = it.get("file_name")
                if fn:
                    drop.add(fn)
            
            cluster_stats[cid] = {
                "total": len(group),
                "kept": kept_count,
                "dropped": len(group) - kept_count
            }
        
        stats["cluster_stats"] = cluster_stats
        stats["num_clusters"] = len(by_cluster)
        return keep, drop, stats

    # Threshold mode
    keep, drop = set(), set()
    scores_kept, scores_dropped = [], []
    
    for it in vl_items:
        fn = it.get("file_name")
        if not fn:
            continue

        cid = it.get("cluster_id")
        score = it.get("uniqueness_score", 1.0)
        
        if cid and cid in dropped_clusters:
            drop.add(fn)
            scores_dropped.append(score)
            continue

        if score >= uniqueness_threshold:
            keep.add(fn)
            scores_kept.append(score)
        else:
            drop.add(fn)
            scores_dropped.append(score)

    stats["avg_score_kept"] = sum(scores_kept) / len(scores_kept) if scores_kept else 0
    stats["avg_score_dropped"] = sum(scores_dropped) / len(scores_dropped) if scores_dropped else 0
    return keep, drop, stats


def iter_train_images(imagenet_root: Path) -> List[Tuple[Path, str, str]]:
    """Returns list of (abs_path, wnid, filename) for ImageNet train."""
    train_root = imagenet_root / "train"
    if not train_root.exists():
        raise FileNotFoundError(f"Missing train folder: {train_root}")

    out = []
    wnid_dirs = [d for d in train_root.iterdir() if d.is_dir()]
    
    iterator = tqdm(wnid_dirs, desc="Scanning train") if HAS_TQDM else wnid_dirs
    for wnid_dir in iterator:
        if not wnid_dir.is_dir():
            continue
        wnid = wnid_dir.name
        for img in wnid_dir.iterdir():
            if img.is_file() and not img.name.startswith('.'):
                out.append((img, wnid, img.name))
    return out


def iter_val_images(imagenet_root: Path) -> List[Tuple[Path, str, str]]:
    """Returns list of (abs_path, wnid, filename) for val."""
    val_root = imagenet_root / "val"
    if not val_root.exists():
        print("Note: No val/ folder found")
        return []
    
    out = []
    sample_dirs = [d for d in val_root.iterdir() if d.is_dir()]
    
    if sample_dirs and len(sample_dirs) > 10 and sample_dirs[0].name.startswith("n"):
        print("Detected organized val/ structure")
        for wnid_dir in sorted(val_root.iterdir()):
            if not wnid_dir.is_dir():
                continue
            wnid = wnid_dir.name
            for img in wnid_dir.iterdir():
                if img.is_file() and not img.name.startswith('.'):
                    out.append((img, wnid, img.name))
    else:
        print("Warning: val/ appears flat - wnids will be UNKNOWN")
        for img in val_root.iterdir():
            if img.is_file() and not img.name.startswith('.'):
                out.append((img, "UNKNOWN", img.name))
    
    return out


def ensure_dir(p: Path) -> None:
    """Create directory if it doesn't exist."""
    p.mkdir(parents=True, exist_ok=True)


def write_manifest_csv(rows: List[Tuple[str, str, str]], out_path: Path) -> None:
    """Write manifest CSV: (split, wnid, rel_path)"""
    ensure_dir(out_path.parent)
    with out_path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["split", "wnid", "rel_path"])
        w.writerows(rows)


def print_statistics(
    kept_manifest, dropped_manifest, keep_set, drop_set, 
    vl_items, policy_stats, unseen_count, config
):
    """Print detailed cleaning statistics."""
    total = len(kept_manifest) + len(dropped_manifest)
    
    print("\n" + "="*70)
    print("CLEANING STATISTICS")
    print("="*70)
    
    print(f"\nVisual Layer Coverage:")
    print(f"  Items loaded:      {len(vl_items):,}")
    print(f"  Unique filenames:  {len(keep_set) + len(drop_set):,}")
    print(f"  Unseen images:     {unseen_count:,} ({'KEPT' if config['keep_unseen'] else 'DROPPED'})")
    
    print(f"\nPolicy Applied:")
    if config['keep_topk_per_cluster']:
        print(f"  Mode:              Top-{config['keep_topk_per_cluster']} per cluster")
        print(f"  Clusters:          {policy_stats.get('num_clusters', 0)}")
        print(f"  Dropped clusters:  {policy_stats.get('clusters_dropped', 0)}")
    else:
        print(f"  Mode:              Uniqueness threshold")
        print(f"  Threshold:         {config['uniqueness_threshold']}")
        if "avg_score_kept" in policy_stats:
            print(f"  Avg score (kept):  {policy_stats['avg_score_kept']:.3f}")
            print(f"  Avg score (drop):  {policy_stats['avg_score_dropped']:.3f}")
    
    print(f"\nDataset Results:")
    print(f"  Total processed:   {total:,}")
    print(f"  ├─ Kept:           {len(kept_manifest):,} ({100*len(kept_manifest)/total:.1f}%)")
    print(f"  └─ Dropped:        {len(dropped_manifest):,} ({100*len(dropped_manifest)/total:.1f}%)")
    
    if vl_items:
        scores = [it.get("uniqueness_score", 0) for it in vl_items]
        print(f"\nUniqueness Distribution (VL items):")
        print(f"  Min:    {min(scores):.3f}")
        print(f"  Median: {sorted(scores)[len(scores)//2]:.3f}")
        print(f"  Max:    {max(scores):.3f}")
    
    print("\n" + "="*70)


# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("ImageNet Cleaning Pipeline")
print("="*70)

# Validate paths
print("\n1. Validating paths...")
if not VL_JSON.exists():
    raise FileNotFoundError(f"VL export not found: {VL_JSON}")
if not IMAGENET_ROOT.exists():
    raise FileNotFoundError(f"ImageNet root not found: {IMAGENET_ROOT}")

print(f"✓ VL JSON:        {VL_JSON}")
print(f"✓ ImageNet root:  {IMAGENET_ROOT}")
print(f"✓ Output root:    {OUT_ROOT}")

# Load dropped clusters if specified
print("\n2. Loading configuration...")
dropped_clusters: Set[str] = set()
if DROP_CLUSTERS_FILE:
    with open(DROP_CLUSTERS_FILE, "r") as f:
        dropped_clusters = {line.strip() for line in f if line.strip()}
    print(f"✓ Loaded {len(dropped_clusters)} clusters to drop")

# Load VL exports
print("\n3. Loading Visual Layer exports...")
vl_items = load_vl_exports([VL_JSON])

# Build policy
print("\n4. Building keep/drop policy...")
keep_set, drop_set, policy_stats = build_keep_set(
    vl_items=vl_items,
    uniqueness_threshold=UNIQUENESS_THRESHOLD,
    keep_topk_per_cluster=KEEP_TOPK_PER_CLUSTER,
    dropped_clusters=dropped_clusters,
)
print(f"✓ Policy: {len(keep_set):,} to keep, {len(drop_set):,} to drop from VL items")

# Build VL seen set
seen_set = {it.get("file_name") for it in vl_items if it.get("file_name")}

# Scan ImageNet
print(f"\n5. Scanning ImageNet...")
train_imgs = iter_train_images(IMAGENET_ROOT)

val_imgs = []
if INCLUDE_VAL:
    val_imgs = iter_val_images(IMAGENET_ROOT)

all_imgs = train_imgs + val_imgs
unseen_count = sum(1 for _, _, fn in all_imgs if fn not in seen_set)

if unseen_count > 0:
    print(f"\n⚠️  Warning: {unseen_count:,} images not in VL export")
    print(f"   Policy: {'KEEPING' if KEEP_UNSEEN else 'DROPPING'} unseen images")

# Process images
print(f"\n6. Processing images...")
if DRY_RUN:
    print("   [DRY RUN MODE - no files will be created]")

kept_manifest = []
dropped_manifest = []

def process_split(imgs, split_name):
    iterator = tqdm(imgs, desc=f"Processing {split_name}") if HAS_TQDM else imgs
    for abs_path, wnid, fn in iterator:
        rel = Path(split_name) / wnid / fn
        
        in_vl = fn in seen_set
        if in_vl:
            keep = fn in keep_set and fn not in drop_set
        else:
            keep = bool(KEEP_UNSEEN)

        if keep:
            kept_manifest.append((split_name, wnid, str(rel)))
            if not DRY_RUN:
                dst = OUT_ROOT / rel
                ensure_dir(dst.parent)
                if MODE == "symlink":
                    if dst.exists():
                        dst.unlink()
                    os.symlink(abs_path, dst)
                else:
                    shutil.copy2(abs_path, dst)
        else:
            dropped_manifest.append((split_name, wnid, str(rel)))

process_split(train_imgs, "train")
if INCLUDE_VAL:
    process_split(val_imgs, "val")

# Write manifests
print(f"\n7. Writing manifests...")
if not DRY_RUN:
    write_manifest_csv(kept_manifest, OUT_ROOT / "manifest_keep.csv")
    write_manifest_csv(dropped_manifest, OUT_ROOT / "manifest_drop.csv")
    print(f"✓ Manifests written to {OUT_ROOT}/")
else:
    print("   [DRY RUN - manifests not written]")

# Print statistics
config = {
    'keep_unseen': KEEP_UNSEEN,
    'keep_topk_per_cluster': KEEP_TOPK_PER_CLUSTER,
    'uniqueness_threshold': UNIQUENESS_THRESHOLD
}

print_statistics(
    kept_manifest, dropped_manifest, keep_set, drop_set,
    vl_items, policy_stats, unseen_count, config
)

if not DRY_RUN:
    print(f"\n✓ SUCCESS! Output dataset created at: {OUT_ROOT}")
    print(f"✓ Mode: {MODE}")
else:
    print(f"\n✓ DRY RUN COMPLETE")
    print(f"  Set DRY_RUN = False to create the actual dataset")
    print(f"  Output will be at: {OUT_ROOT}")

ImageNet Cleaning Pipeline

1. Validating paths...
✓ VL JSON:        /Users/saeedarellano/Desktop/imagenet_jsons/metadata 11.json
✓ ImageNet root:  /Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC
✓ Output root:    /Users/saeedarellano/visual_layer/capstone_project_Visual-Layer/clean

2. Loading configuration...

3. Loading Visual Layer exports...
Loading metadata 11.json...
✓ Loaded 162 total items from 1 file(s)

4. Building keep/drop policy...
✓ Policy: 0 to keep, 162 to drop from VL items

5. Scanning ImageNet...


Scanning train:   0%|          | 0/512 [00:00<?, ?it/s]


⚠️  Warning: 656,348 images not in VL export
   Policy: KEEPING unseen images

6. Processing images...
   [DRY RUN MODE - no files will be created]


Processing train:   0%|          | 0/655570 [00:00<?, ?it/s]

Processing val:   0%|          | 0/843 [00:00<?, ?it/s]


7. Writing manifests...
   [DRY RUN - manifests not written]

CLEANING STATISTICS

Visual Layer Coverage:
  Items loaded:      162
  Unique filenames:  162
  Unseen images:     656,348 (KEPT)

Policy Applied:
  Mode:              Uniqueness threshold
  Threshold:         999
  Avg score (kept):  0.000
  Avg score (drop):  0.370

Dataset Results:
  Total processed:   656,413
  ├─ Kept:           656,348 (100.0%)
  └─ Dropped:        65 (0.0%)

Uniqueness Distribution (VL items):
  Min:    0.002
  Median: 0.246
  Max:    0.999


✓ DRY RUN COMPLETE
  Set DRY_RUN = False to create the actual dataset
  Output will be at: /Users/saeedarellano/visual_layer/capstone_project_Visual-Layer/clean


Seeing how many images, local imagenet has

In [7]:
from pathlib import Path
from typing import List, Tuple

def scan_imagenet_train_val(imagenet_root: Path, include_val: bool = True) -> List[Tuple[Path, str, str, str]]:
    """
    Returns (abs_path, split, wnid, filename) for ImageNet.
    - Train: expects train/<wnid>/*.JPEG
    - Val: supports either val/<wnid>/*.JPEG OR flat val/*.JPEG
    Read-only: does not modify anything.
    """
    exts = {".jpeg", ".jpg", ".png"}
    out: List[Tuple[Path, str, str, str]] = []

    train_root = imagenet_root / "train"
    if not train_root.exists():
        raise FileNotFoundError(f"Missing train folder: {train_root}")

    # ---- TRAIN: train/<wnid>/* ----
    for wnid_dir in train_root.iterdir():
        if not wnid_dir.is_dir():
            continue
        wnid = wnid_dir.name
        for p in wnid_dir.iterdir():
            if p.is_file() and p.suffix.lower() in exts:
                out.append((p, "train", wnid, p.name))

    # ---- VAL: either val/<wnid>/* or flat val/* ----
    if include_val:
        val_root = imagenet_root / "val"
        if not val_root.exists():
            raise FileNotFoundError(f"Missing val folder: {val_root}")

        val_dirs = [d for d in val_root.iterdir() if d.is_dir()]
        looks_wnid = bool(val_dirs) and all(d.name.startswith("n") for d in val_dirs[:10])

        if looks_wnid:
            for wnid_dir in val_dirs:
                wnid = wnid_dir.name
                for p in wnid_dir.iterdir():
                    if p.is_file() and p.suffix.lower() in exts:
                        out.append((p, "val", wnid, p.name))
        else:
            # flat val/
            for p in val_root.iterdir():
                if p.is_file() and p.suffix.lower() in exts:
                    out.append((p, "val", "UNKNOWN", p.name))

    return out



In [8]:
IMAGENET_ROOT = Path("/Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC")
imgs = scan_imagenet_train_val(IMAGENET_ROOT, include_val=True)

print("Total:", len(imgs))
print("Train:", sum(1 for _, s, _, _ in imgs if s == "train"))
print("Val:", sum(1 for _, s, _, _ in imgs if s == "val"))
print("First 3:", imgs[:3])


Total: 656413
Train: 655570
Val: 843
First 3: [(PosixPath('/Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train/n02795169/n02795169_22072.JPEG'), 'train', 'n02795169', 'n02795169_22072.JPEG'), (PosixPath('/Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train/n02795169/n02795169_5580.JPEG'), 'train', 'n02795169', 'n02795169_5580.JPEG'), (PosixPath('/Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train/n02795169/n02795169_14510.JPEG'), 'train', 'n02795169', 'n02795169_14510.JPEG')]


Total images found: 0
